# Introduction

This Notebook introduces user-based collaborative filtering.

# User-item interaction matrix

In [1]:
import torch

# Rows = users, Columns = items (e.g., movies)
# 0 means "no interaction / not rated"
ratings = torch.tensor([
    [5.0, 4.0, 0.0, 1.0, 0.0],
    [4.0, 5.0, 0.0, 1.0, 0.0],
    [1.0, 0.0, 5.0, 4.0, 0.0],
    [0.0, 1.0, 4.0, 5.0, 0.0],
    [5.0, 4.0, 0.0, 0.0, 3.0],
])

# Users similarity

In [2]:
import torch.nn.functional as F

# Normalize user vectors (L2 norm)
normalized_ratings = F.normalize(ratings, p=2, dim=1)

# Compute cosine similarity between users
user_similarity = normalized_ratings @ normalized_ratings.T

print(user_similarity)


tensor([[1.0000, 0.9762, 0.2143, 0.2143, 0.8947],
        [0.9762, 1.0000, 0.1905, 0.2381, 0.8729],
        [0.2143, 0.1905, 1.0000, 0.9524, 0.1091],
        [0.2143, 0.2381, 0.9524, 1.0000, 0.0873],
        [0.8947, 0.8729, 0.1091, 0.0873, 1.0000]])


# User similarities

In [3]:
target_user = 0

similarities = user_similarity[target_user].clone()

# Remove self-similarity
similarities[target_user] = 0

print(similarities)

tensor([0.0000, 0.9762, 0.2143, 0.2143, 0.8947])


# Predicted ratings

In [4]:
# Weighted sum of ratings from similar users
weighted_ratings = similarities @ ratings

# Normalize by total similarity
predicted_ratings = weighted_ratings / similarities.sum()

print(predicted_ratings)

tensor([3.7368, 3.7722, 0.8387, 1.2632, 1.1673])


# Turn predictions into recommendations

In [5]:
# Identify unrated items
unrated = ratings[target_user] == 0

# Mask already-rated items
recommendation_scores = predicted_ratings.clone()
recommendation_scores[~unrated] = -1

# Select best recommendation
recommended_item = torch.argmax(recommendation_scores).item()

print(f"Recommended item: {recommended_item}")

Recommended item: 4


# Choosing a similarity measure

In [6]:
# Subtract user mean (centering)
user_means = ratings.sum(dim=1, keepdim=True) / (ratings != 0).sum(dim=1, keepdim=True)
centered = ratings - user_means

centered[ratings == 0] = 0  # keep missing values at zero

normalized_centered = F.normalize(centered, p=2, dim=1)
pearson_similarity = normalized_centered @ normalized_centered.T


In [7]:
pearson_similarity

tensor([[ 1.0000,  0.8846, -0.6282, -0.6282,  0.4003],
        [ 0.8846,  1.0000, -0.3590, -0.8974,  0.1601],
        [-0.6282, -0.3590,  1.0000,  0.2564, -0.5604],
        [-0.6282, -0.8974,  0.2564,  1.0000,  0.0000],
        [ 0.4003,  0.1601, -0.5604,  0.0000,  1.0000]])